
# Fine-tuning (BF16, minimal, in-notebook only)
- Không chỉnh sửa file `.py` bên ngoài.
- Cấu hình **BF16** (ổn định, không cần GradScaler), tránh lỗi *"Attempting to unscale FP16 gradients"*.


In [2]:
# import
import os, json, random, pickle
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from ConversationItems import ConversationItem
import ConversationItems as CI
import gc



CUDA devices: 1


In [3]:
# Config
BASE_MODEL   = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR   = "./ft-qwen2.5-0.5B-sft"
MAX_SEQ_LEN  = 2048
USE_4BIT     = False
SEED         = 42

BATCH_SIZE   = 2
GRAD_ACCUM   = 4
LR           = 2e-5
NUM_EPOCHS   = 1
EVAL_STEPS   = 200
SAVE_STEPS   = 400
LOG_STEPS    = 20
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.0

# BF16 ON, FP16 OFF
FP16         = False
BF16         = True

random.seed(SEED)
torch.manual_seed(SEED)

# Tăng tốc trên NVIDIA
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("CUDA devices:", torch.cuda.device_count())


In [4]:

# Load pickle
def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

raw_train_obj = load_pickle("data/train.pkl")
raw_eval_obj  = load_pickle("data/test.pkl")

def to_list_of_dicts(obj, label="train"):
    if isinstance(obj, list) and all(isinstance(x, dict) for x in obj):
        return obj
    try:
        from datasets import Dataset as HFDataset
        if isinstance(obj, HFDataset):
            return [obj[i] for i in range(len(obj))]
    except Exception:
        pass
    try:
        import pandas as pd
        if isinstance(obj, pd.DataFrame):
            return obj.to_dict(orient="records")
    except Exception:
        pass
    if isinstance(obj, dict):
        for k in ("data","rows","items"):
            if k in obj and isinstance(obj[k], list):
                return obj[k]
    if isinstance(obj, list) and all(isinstance(x, (list, tuple)) for x in obj):
        out = []
        for i, pair in enumerate(obj):
            if len(pair) >= 2:
                u, a = pair[0], pair[1]
                out.append({
                    "id": f"{label}_{i}",
                    "conversations": [
                        {"from":"human", "value": str(u)},
                        {"from":"assistant", "value": str(a)}
                    ]
                })
        if out: return out
    if isinstance(obj, list) and len(obj) > 0:
        tmp = []
        for x in obj:
            if isinstance(x, dict):
                tmp.append(x)
            elif hasattr(x, "__dict__"):
                tmp.append(dict(x.__dict__))
        if tmp: return tmp
    raise TypeError("Expect list[dict] | HF Dataset | DataFrame | list[(in,out)] | dict wrapper.")

train_list = to_list_of_dicts(raw_train_obj, "train")
eval_list  = to_list_of_dicts(raw_eval_obj,  "eval")
print(f"Loaded (raw): train={len(train_list)}, eval={len(eval_list)}")

def coerce_record(rec, idx_prefix="rec"):
    out = {"id": str(rec.get("id") or f"{idx_prefix}_anon")}
    if isinstance(rec.get("conversations"), list):
        out["conversations"] = rec["conversations"]; return out
    if isinstance(rec.get("messages"), list):
        conv = []
        for m in rec["messages"]:
            frm = (m.get("from") or m.get("role") or "").lower()
            val = m.get("value") or m.get("content") or ""
            if val:
                conv.append({"from": frm, "value": str(val)})
        out["conversations"] = conv; return out
    for u,a in (("input","output"),("prompt","response"),("question","answer"),("user","assistant")):
        if u in rec and a in rec:
            out["conversations"] = [
                {"from":"human","value":str(rec[u])},
                {"from":"assistant","value":str(rec[a])},
            ]; return out
    if "text" in rec:
        out["conversations"] = [{"from":"human","value":str(rec["text"])}]; return out
    out["conversations"] = []; return out

train_std = [coerce_record(r, "train") for r in train_list]
eval_std  = [coerce_record(r, "eval")  for r in eval_list]
train_std = [r for r in train_std if r["conversations"]]
eval_std  = [r for r in eval_std  if r["conversations"]]
print(f"Standardized: train={len(train_std)}, eval={len(eval_std)}")


Loaded (raw): train=11111, eval=1235
Standardized: train=11111, eval=1235


In [5]:

# Nới ngưỡng lọc trong notebook
CI.MIN_TOKENS = 32
CI.MIN_CHARS  = 10
CI.MAX_CHARS  = 9000
CI.MAX_TOKENS = MAX_SEQ_LEN

print("Thresholds in use:", CI.MIN_TOKENS, CI.MIN_CHARS, CI.MAX_CHARS, CI.MAX_TOKENS)


Thresholds in use: 32 10 9000 2048


In [6]:

# Normalize bằng ConversationItem -> SFT messages
def to_sft(items):
    sft = []
    for dp in items:
        try:
            it = ConversationItem(dp)
            if it.include:
                sft.append(it.to_sft())
        except Exception:
            pass
    return sft

train_sft = to_sft(train_std)
eval_sft  = to_sft(eval_std)
print(f"SFT: train={len(train_sft)}, eval={len(eval_sft)}")
print(json.dumps(train_sft[0], ensure_ascii=False, indent=2) if train_sft else "No sample")


SFT: train=11111, eval=1235
{
  "id": "12077",
  "messages": [
    {
      "role": "user",
      "content": "Chủ đề chính của vở kịch \"Romeo và Juliet\" là gì"
    },
    {
      "role": "assistant",
      "content": "Chủ đề chính của vở kịch Romeo và Juliet là sức mạnh của tình yêu để vượt qua trở ngại."
    },
    {
      "role": "user",
      "content": "Bạn có thể đưa ra một số ví dụ cụ thể về những trở ngại mà Romeo và Juliet đã vượt qua được không?"
    },
    {
      "role": "assistant",
      "content": "Chắc chắn! Dưới đây là một số ví dụ cụ thể về những trở ngại mà Romeo và Juliet phải đối mặt trong vở kịch: - Họ đến từ những gia đình thù địch, nhà Capulets và nhà Montagues, những người có mối thù truyền kiếp có nguy cơ khiến họ xa cách. - Cha mẹ của Juliet đã hứa gả cô cho Bá tước Paris, vì vậy cô và Romeo phải vượt qua sự phản đối của họ để được ở bên nhau. - Khi Romeo giết Tybalt, em họ của Juliet, anh ta bị trục xuất khỏi Verona, khiến việc ở bên nhau của họ dường như cà

In [7]:

# Apply chat template -> raw text
tokenizer = ConversationItem.tokenizer
tokenizer.model_max_length = MAX_SEQ_LEN

def to_text(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_texts = [to_text(x["messages"]) for x in train_sft]
eval_texts  = [to_text(x["messages"]) for x in eval_sft]
print("Texts:", len(train_texts), len(eval_texts))
print("Sample:", (train_texts[0][:200]+"...") if train_texts else "EMPTY")


Texts: 11111 1235
Sample: <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Chủ đề chính của vở kịch "Romeo và Juliet" là gì<|im_end|>
<|im_start|>assistant
Chủ ...


In [8]:

# Chunk text dài theo token
CHUNK_TOKENS  = MAX_SEQ_LEN
CHUNK_OVERLAP = 50

def chunk_texts_by_tokens(texts, tokenizer, max_len=CHUNK_TOKENS, overlap=CHUNK_OVERLAP):
    out = []
    for t in texts:
        ids = tokenizer(t, add_special_tokens=False, return_attention_mask=False)["input_ids"]
        if len(ids) <= max_len:
            out.append(t); continue
        start = 0; step = max_len - overlap
        while start < len(ids):
            window = ids[start:start+max_len]
            if not window: break
            out.append(tokenizer.decode(window, skip_special_tokens=False))
            start += step
    return out

train_texts = chunk_texts_by_tokens(train_texts, tokenizer)
eval_texts  = chunk_texts_by_tokens(eval_texts,  tokenizer)
print("After chunk:", len(train_texts), len(eval_texts))


Token indices sequence length is longer than the specified maximum sequence length for this model (2063 > 2048). Running this sequence through the model will result in indexing errors


After chunk: 11306 1254


In [9]:

# Build HF Datasets & tokenize
train_ds = Dataset.from_dict({"text": train_texts})
eval_ds  = Dataset.from_dict({"text": eval_texts})

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=tokenizer.model_max_length,
        padding=False,
        add_special_tokens=False,
        return_attention_mask=True,
    )

train_tok = train_ds.map(tokenize_function, batched=True, remove_columns=["text"])
eval_tok  = eval_ds.map(tokenize_function,  batched=True, remove_columns=["text"])

print("Columns:", train_tok.column_names)
if len(train_tok) == 0:
    raise ValueError("train_tok rỗng sau tokenize. Kiểm tra lại ngưỡng lọc hoặc dữ liệu.")


Map:   0%|          | 0/11306 [00:00<?, ? examples/s]

Map:   0%|          | 0/1254 [00:00<?, ? examples/s]

Columns: ['input_ids', 'attention_mask']


In [10]:

# Load model và đồng bộ special tokens
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try: del model
except: pass
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    device_map=None,
    low_cpu_mem_usage=True,
    dtype=(torch.bfloat16 if BF16 and torch.cuda.is_available() else None),
)
model.to(DEVICE)

# Đồng bộ special tokens + pad
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

if tokenizer.eos_token_id is not None:
    model.config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id

if tokenizer.bos_token_id is not None:
    model.config.bos_token_id = tokenizer.bos_token_id
    model.generation_config.bos_token_id = tokenizer.bos_token_id

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("Model loaded on", DEVICE, "| dtype:", model.dtype)


Model loaded on cuda | dtype: torch.bfloat16


In [10]:

#Trainer (eval_strategy compat + remove_unused_columns=False)
import transformers
from packaging.version import parse as V

os.makedirs(OUTPUT_DIR, exist_ok=True)

ta_kwargs = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=NUM_EPOCHS,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOG_STEPS,
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    report_to="none",
    fp16=False,                 # tắt FP16
    bf16=True,                  # bật BF16
    gradient_checkpointing=True,
    remove_unused_columns=False,
    optim="adamw_torch_fused",
)

if V(transformers.__version__) >= V("4.46"):
    ta_kwargs["eval_strategy"] = "steps"
else:
    ta_kwargs["evaluation_strategy"] = "steps"

args = TrainingArguments(**ta_kwargs)

# Tránh cảnh báo use_cache với gradient checkpointing
if getattr(args, "gradient_checkpointing", False):
    model.config.use_cache = False

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok if len(eval_tok) > 0 else None,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training complete ->", OUTPUT_DIR)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss
200,1.505600,1.496541
400,1.451100,1.452035
600,1.467100,1.429075
800,1.447000,1.417013
1000,1.388900,1.412960
1200,1.414300,1.412194
1400,1.439500,1.412079


Training complete -> ./ft-qwen2.5-0.5B-sft


In [11]:

# Quick inference
from transformers import TextStreamer

messages = [{"role":"user","content":"Chào bạn, mình test model sau fine-tune nha?"}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
streamer = TextStreamer(tokenizer, skip_special_tokens=True)
_ = model.generate(inputs, max_new_tokens=128, temperature=0.7, top_p=0.9, do_sample=True, streamer=streamer)
print("\n--- Done.")


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Chào bạn, mình test model sau fine-tune nha?
assistant
Xin chào! Tôi có thể giúp gì cho bạn trong quá trình fine-tuning một mô hình? Tôi sẽ cố gắng hỗ trợ bạn với mọi thông tin và câu hỏi cần thiết.

--- Done.


In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./ft-qwen2.5-0.5B-sft"  # không cần vào checkpoint-1414
tok = AutoTokenizer.from_pretrained(model_path, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
).eval()

# Nếu có chat_template.jinja, hãy dùng nó:
messages = [
    {"role": "system", "content": "Bạn là trợ lý tư vấn giáo dục thân thiện."},
    {"role": "user", "content": "Chào bạn, học đại học có khó không?"}
]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tok(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True, temperature=0.7, top_p=0.9,
        repetition_penalty=1.05
    )

print(tok.decode(out[0], skip_special_tokens=True))


system
Bạn là trợ lý tư vấn giáo dục thân thiện.
user
Chào bạn, học đại học có khó không?
assistant
Là một người mẫu ngôn ngữ AI, tôi không có kinh nghiệm cá nhân hay trải nghiệm cá nhân. Tuy nhiên, theo các chuyên gia trong lĩnh vực đại học, có thể nói rằng đại học là một trong những điều quan trọng nhất trong cuộc đời của con người và nó mang lại cho chúng nhiều cơ hội và kiến thức bổ ích hơn.


In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM

src = "./ft-qwen2.5-0.5B-sft/checkpoint-1414"
dst = "./export-qwen2.5-0.5B-sft"

tok = AutoTokenizer.from_pretrained(src, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(src)

model.save_pretrained(dst, safe_serialization=True)
tok.save_pretrained(dst)

print("Saved to:", dst)


Saved to: ./export-qwen2.5-0.5B-sft
